# Evaluation Robustness and Goodhart's Law

When a metric becomes a target, it ceases to be a good metric. This is Goodhart's Law, and it has played out repeatedly in NLP evaluation. Understanding how benchmarks get gamed — and how to build more robust evaluation infrastructure — is the meta-skill that keeps evaluation meaningful.

## Prompt Sensitivity: A First-Order Robustness Check

Before worrying about deliberate gaming, quantify how much your evaluation results depend on arbitrary evaluation choices:
- Does changing the prompt template by one word change accuracy by more than 5%?
- Does changing the order of few-shot examples matter?
- Does the answer choice ordering for MCQ tasks affect results?

If yes to any of these, your benchmark is measuring something other than the capability you care about. Robust evaluation should show stable results across these surface variations.

In [ ]:
import random

def evaluate_prompt_sensitivity(model_fn, question: str, answer: str, templates: list) -> dict:
    results = []
    for tmpl in templates:
        prompt = tmpl.format(question=question)
        response = model_fn(prompt)
        correct = answer.lower() in response.lower()
        results.append({'template': tmpl[:40], 'response': response[:60], 'correct': correct})
    accuracy = sum(r['correct'] for r in results) / len(results)
    return {'accuracy': accuracy, 'results': results,
            'sensitive': max(r['correct'] for r in results) != min(r['correct'] for r in results)}

templates = [
    'Answer the following: {question}',
    'Question: {question}\nAnswer:',
    'Please respond to: {question}',
    '{question}',
    'As an expert, answer: {question}',
]

def inconsistent_model(prompt):
    # Sensitive to phrasing
    if 'Answer the following' in prompt or 'Please respond' in prompt:
        return 'Paris'
    return 'London'

sensitivity = evaluate_prompt_sensitivity(
    inconsistent_model,
    question='What is the capital of France?',
    answer='Paris',
    templates=templates
)
print(f'Accuracy across templates: {sensitivity["accuracy"]:.0%}')
print(f'Sensitive to phrasing: {sensitivity["sensitive"]}')
print()
for r in sensitivity['results']:
    status = 'OK' if r['correct'] else 'FAIL'
    print(f'[{status}] {r["template"]:<45} -> {r["response"]}')

## Benchmark Gaming in Practice

How benchmarks have been gamed in the ML community:

**GLUE/SuperGLUE**: models fine-tuned directly on dev set examples; some leaderboard entries used data augmentation that introduced target leakage.

**MMLU**: models trained specifically on MMLU question formats via synthetic data; contamination from web-crawled study guides.

**HumanEval**: models trained on GitHub code that includes HumanEval solutions in comments or documentation.

**SWE-bench**: agent frameworks that submit outputs without actually solving the underlying issue, gaming the test suite evaluation.

In each case, optimization pressure was applied directly to the metric rather than to the underlying capability.

## Goodhart's Law in Evaluation Design

Goodhart's Law: when a measure becomes a target, it ceases to be a good measure.

In LLM evaluation, this manifests as:
1. **Benchmark-specific training**: MMLU performance improves faster than reasoning ability because models are trained on MMLU-format questions
2. **Judge gaming**: models learn to produce outputs that LLM judges prefer (verbose, confident, well-formatted) regardless of accuracy
3. **Length gaming**: RLHF with preference data causes models to produce longer responses because annotators exhibit length bias
4. **Refusal calibration gaming**: models learn to pattern-match refusal triggers rather than evaluate underlying harm

The mitigation is constant benchmark rotation, held-out eval sets, and human preference data that is explicitly de-biased for length and style.

In [ ]:
def benchmark_rotation_check(model_results: dict, correlation_threshold: float = 0.85) -> dict:
    benchmarks = list(model_results.keys())
    n = len(benchmarks)
    if n < 2:
        return {'warning': 'Need at least 2 benchmarks to check correlation'}
    # Compute pairwise rank correlation
    def rank(scores):
        sorted_models = sorted(scores.items(), key=lambda x: -x[1])
        return {m: i for i, (m, _) in enumerate(sorted_models)}
    ranks = {b: rank(model_results[b]) for b in benchmarks}
    models_list = list(list(model_results.values())[0].keys())
    correlations = {}
    for i in range(n):
        for j in range(i+1, n):
            b1, b2 = benchmarks[i], benchmarks[j]
            r1 = [ranks[b1][m] for m in models_list]
            r2 = [ranks[b2][m] for m in models_list]
            mean1, mean2 = sum(r1)/len(r1), sum(r2)/len(r2)
            num = sum((r1[k]-mean1)*(r2[k]-mean2) for k in range(len(r1)))
            d1 = (sum((r-mean1)**2 for r in r1))**0.5
            d2 = (sum((r-mean2)**2 for r in r2))**0.5
            corr = num/(d1*d2) if d1*d2 > 0 else 0
            correlations[f'{b1} vs {b2}'] = round(corr, 3)
    high_corr = {k:v for k,v in correlations.items() if v > correlation_threshold}
    return {'correlations': correlations, 'high_correlation_pairs': high_corr,
            'warning': 'Highly correlated benchmarks may be measuring the same thing' if high_corr else None}

model_scores = {
    'MMLU': {'ModelA': 0.87, 'ModelB': 0.84, 'ModelC': 0.79},
    'ARC': {'ModelA': 0.83, 'ModelB': 0.81, 'ModelC': 0.75},
    'HellaSwag': {'ModelA': 0.91, 'ModelB': 0.89, 'ModelC': 0.85},
    'Math': {'ModelA': 0.52, 'ModelB': 0.71, 'ModelC': 0.48},  # different ranking!
}

check = benchmark_rotation_check(model_scores)
print('Pairwise rank correlations:')
for pair, corr in check['correlations'].items():
    print(f'  {pair}: {corr}')
if check['warning']:
    print(f'\nWarning: {check["warning"]}')
    print('High correlation pairs:', check['high_correlation_pairs'])

## Building Robust Evaluation Infrastructure

Practical guidelines:
1. **Never report a single number**: report mean and variance across seeds, templates, and example orderings
2. **Separate leaderboard and development sets**: never optimize directly against your public leaderboard metric
3. **Rotate benchmarks regularly**: once a benchmark saturates (top models all within 2%), retire it
4. **Mix automatic and human evaluation**: automatic metrics can be gamed; human preference is harder to game but noisier
5. **Measure calibration, not just accuracy**: a well-calibrated 80% model is more trustworthy than an uncalibrated 85% model
6. **Track metric-to-metric consistency**: if your model's MMLU improves but Arena ELO stays flat, something is wrong